# Challenge: build it from scratch

Time to put Class 2 into practice. **You** pick a dataset, build the whole training process yourself, run real experiments, and get the best result you can.

Rules of the game:
1. Start from a **baseline** – the number you have to beat.
2. Change **one thing at a time** and write down what happened (remember the [experiments notebook](../../Class-2/code/experiments.ipynb)).
3. Use the **validation set** for all your decisions. The **test set** is touched **once**, at the very end.
4. Every result is **mean ± std over a few seeds**, not one lucky run.

What you hand in (or show in class): the results table, your best configuration, the final test score, and 3–5 sentences on what worked and what didn't.

Everything here runs on a laptop CPU in a few minutes per experiment.

## 0. Setup

In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import time
import numpy as np
import matplotlib.pyplot as plt
import keras

print("Keras", keras.__version__, "on", keras.backend.backend())

## 1. Pick your dataset

All of them are built into Keras and download automatically.

| `DATASET` | Task | What it is | Size | Difficulty |
|---|---|---|---|---|
| `"fashion_mnist"` | 10-class classification | 28×28 grayscale photos of clothes (shirts, shoes, bags…) | 60 000 + 10 000 | ★★ |
| `"california_housing"` | Regression | Predict the median house price of a district from 8 features | 20 640 | ★★ |
| `"reuters"` | 46-class classification | Newswires labelled with their topic (bag of words) | 8 982 + 2 246 | ★★★ |
| `"cifar10"` | 10-class classification | 32×32 colour photos (planes, cats, trucks…) – hard without convolutions! | 50 000 + 10 000 | ★★★★ |

Change the line below and re-run the notebook from here.

In [ ]:
DATASET = "fashion_mnist"   # "fashion_mnist" | "california_housing" | "reuters" | "cifar10"

SPLIT_SEED = 2026            # fixed: the train/validation split never changes
VAL_FRACTION = 0.15

In [ ]:
def multi_hot(sequences, dimension=10_000):
    out = np.zeros((len(sequences), dimension), dtype="float32")
    for i, seq in enumerate(sequences):
        out[i, seq] = 1.0
    return out


def load_dataset(name):
    """Returns raw arrays plus a small description of the task."""
    if name == "fashion_mnist":
        (x, y), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
        task, n_classes = "classification", 10
    elif name == "cifar10":
        (x, y), (x_test, y_test) = keras.datasets.cifar10.load_data()
        y, y_test = y.ravel(), y_test.ravel()
        task, n_classes = "classification", 10
    elif name == "reuters":
        (x, y), (x_test, y_test) = keras.datasets.reuters.load_data(num_words=10_000)
        x, x_test = multi_hot(x), multi_hot(x_test)
        task, n_classes = "classification", 46
    elif name == "california_housing":
        (x, y), (x_test, y_test) = keras.datasets.california_housing.load_data(version="large")
        task, n_classes = "regression", None
    else:
        raise ValueError(f"Unknown dataset: {name}")
    return x, y, x_test, y_test, task, n_classes


x_full, y_full, x_test_raw, y_test_raw, TASK, N_CLASSES = load_dataset(DATASET)

# Fixed train / validation split (the test set is kept aside)
rng = np.random.default_rng(SPLIT_SEED)
idx = rng.permutation(len(x_full))
n_val = int(len(idx) * VAL_FRACTION)
x_train_raw, y_train = x_full[idx[n_val:]], y_full[idx[n_val:]]
x_val_raw, y_val = x_full[idx[:n_val]], y_full[idx[:n_val]]

print(f"{DATASET}: {TASK}, train {x_train_raw.shape}, val {x_val_raw.shape}, test {x_test_raw.shape}")

## 2. Look at the data first

Before any model: what do the inputs look like? Are the classes balanced? What's the range of the target?

**Your turn:** add your own exploration below – histograms, example images, min/max of features. Write down one thing that surprised you.

In [ ]:
if TASK == "classification":
    counts = np.bincount(y_train, minlength=N_CLASSES)
    print("Examples per class (train):", counts)
    print(f"Smallest class: {counts.min()}, largest class: {counts.max()}")
else:
    print(f"Target: min {y_train.min():.0f}, mean {y_train.mean():.0f}, max {y_train.max():.0f}")
    print("Feature means:", np.round(x_train_raw.mean(axis=0), 2))
    print("Feature stds: ", np.round(x_train_raw.std(axis=0), 2))

In [ ]:
if DATASET in ("fashion_mnist", "cifar10"):
    plt.figure(figsize=(8, 2))
    for i in range(8):
        plt.subplot(1, 8, i + 1)
        plt.imshow(x_train_raw[i], cmap="binary" if DATASET == "fashion_mnist" else None)
        plt.title(int(y_train[i]))
        plt.axis("off")
    plt.show()
elif TASK == "regression":
    plt.hist(y_train, bins=50)
    plt.title("Distribution of the target")
    plt.show()

## 3. The baseline – the number to beat

A "model" that doesn't learn anything:
- **classification** → always predict the most common class,
- **regression** → always predict the mean of the training targets.

If your network doesn't clearly beat this, something is wrong (Class 2, "Know your baseline").

In [ ]:
if TASK == "classification":
    majority = np.bincount(y_train).argmax()
    BASELINE = np.mean(y_val == majority)
    print(f"Baseline validation accuracy (always class {majority}): {BASELINE:.3f}")
    print(f"Random guessing would give about {1 / N_CLASSES:.3f}")
else:
    BASELINE = np.mean(np.abs(y_val - y_train.mean()))
    print(f"Baseline validation MAE (always predict the mean): {BASELINE:,.0f}")

## 4. Preprocessing

A working default is below so the notebook runs end to end – **but this is your first place to improve.**

Ideas: images – scale to [0, 1] (done) or standardize; housing – standardize features (done), scale the target (done – why does it help?), maybe log-transform some features; reuters – try fewer words (`num_words`).

Remember: every statistic (mean, std…) is computed **on the training set only**.

In [ ]:
def preprocess(x_train, x_val, x_test):
    if DATASET in ("fashion_mnist", "cifar10"):
        f = lambda x: (x.reshape(len(x), -1) / 255.0).astype("float32")
        return f(x_train), f(x_val), f(x_test)
    if DATASET == "california_housing":
        mean, std = x_train.mean(axis=0), x_train.std(axis=0)
        f = lambda x: ((x - mean) / std).astype("float32")
        return f(x_train), f(x_val), f(x_test)
    return x_train, x_val, x_test   # reuters is already multi-hot


x_train, x_val, x_test = preprocess(x_train_raw, x_val_raw, x_test_raw)

# Regression: scale the target down (and back up when reporting)
TARGET_SCALE = 100_000 if TASK == "regression" else 1
y_train_s, y_val_s = y_train / TARGET_SCALE, y_val / TARGET_SCALE

print("Model input shape:", x_train.shape[1:])

## 5. Model and training

Everything you can change lives in one `CONFIG` dictionary – that's what you'll vary in your experiments.

The last layer, loss and metric are chosen from the task – check them against the cheat sheet in [Class 2](../../Class-2/Class-2.md).

In [ ]:
CONFIG = {
    "hidden_units": [64, 32],   # one number per hidden layer
    "activation": "relu",
    "dropout": 0.0,
    "learning_rate": 1e-3,
    "batch_size": 128,
    "epochs": 30,               # upper limit – EarlyStopping usually stops earlier
    "patience": 4,
}
SEEDS = [1, 2, 3]

METRIC = "accuracy" if TASK == "classification" else "mae"
HIGHER_IS_BETTER = TASK == "classification"
FMT = ".4f" if TASK == "classification" else ",.0f"   # how to print scores


def build_model(config):
    model = keras.Sequential([keras.Input(shape=x_train.shape[1:])])
    for units in config["hidden_units"]:
        model.add(keras.layers.Dense(units, activation=config["activation"]))
        if config["dropout"] > 0:
            model.add(keras.layers.Dropout(config["dropout"]))
    if TASK == "classification":
        model.add(keras.layers.Dense(N_CLASSES, activation="softmax"))
        loss = "sparse_categorical_crossentropy"
    else:
        model.add(keras.layers.Dense(1))
        loss = "mse"
    model.compile(optimizer=keras.optimizers.Adam(config["learning_rate"]),
                  loss=loss, metrics=[METRIC])
    return model


build_model(CONFIG).summary()

In [ ]:
RESULTS = []      # every experiment ends up here
HISTORIES = {}


def run_once(config, seed):
    keras.utils.set_random_seed(seed)
    model = build_model(config)
    start = time.perf_counter()
    history = model.fit(
        x_train, y_train_s,
        validation_data=(x_val, y_val_s),
        epochs=config["epochs"],
        batch_size=config["batch_size"],
        callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=config["patience"],
                                                 restore_best_weights=True)],
        verbose=0,
    )
    _, val_metric = model.evaluate(x_val, y_val_s, verbose=0)
    if TASK == "regression":
        val_metric *= TARGET_SCALE
    return val_metric, len(history.history["loss"]), time.perf_counter() - start, history


def experiment(name, **changes):
    """Run CONFIG + your changes over all SEEDS and log the result."""
    config = {**CONFIG, **changes}
    scores, epochs, seconds = [], [], []
    for seed in SEEDS:
        score, n_epochs, t, history = run_once(config, seed)
        scores.append(score); epochs.append(n_epochs); seconds.append(t)
    HISTORIES[name] = history
    row = {"name": name, "changes": changes, "mean": np.mean(scores), "std": np.std(scores, ddof=1),
           "epochs": np.mean(epochs), "seconds": np.mean(seconds), "config": config}
    RESULTS.append(row)
    print(f"{name:<28} val {METRIC}: {row['mean']:{FMT}} ± {row['std']:{FMT}}   "
          f"(~{row['epochs']:.0f} epochs, {row['seconds']:.0f}s per run)")


def show_results():
    rows = sorted(RESULTS, key=lambda r: r["mean"], reverse=HIGHER_IS_BETTER)
    print(f"Baseline: {BASELINE:{FMT}}\n")
    print(f"{'experiment':<28} {'val ' + METRIC:>16}   changes")
    for r in rows:
        print(f"{r['name']:<28} {r['mean']:>10{FMT}} ± {r['std']:{FMT}}   {r['changes']}")


def plot_history(name):
    h = HISTORIES[name].history
    plt.plot(h["loss"], "bo", label="train loss")
    plt.plot(h["val_loss"], "b", label="validation loss")
    plt.title(name); plt.xlabel("epoch"); plt.legend(); plt.show()

## 6. Experiments

First run the default configuration. Then **change one thing per experiment** and give it a clear name.

Ideas, roughly in the order worth trying:
1. Learning rate: `1e-2`, `3e-4`
2. Size: `hidden_units=[128, 64]`, `[256]`, `[512, 256, 128]`
3. Regularization: `dropout=0.3`
4. Batch size: `32`, `512`
5. Activation: `"tanh"`, `"gelu"`
6. Preprocessing changes (go back to section 4, re-run, then run your experiment again)

In [ ]:
experiment("default")
plot_history("default")

In [ ]:
# Your experiments – one change each. Two examples to start:
experiment("lr 3e-4", learning_rate=3e-4)
experiment("bigger [128, 64]", hidden_units=[128, 64])

In [ ]:
# ...add more here...

In [ ]:
show_results()

## 7. The final test – only once!

Pick your best configuration **based on validation only**, train it again on the training set and evaluate it on the test set. This is the number you report.

If the test score is much worse than validation, you've probably over-tuned to the validation set – worth a sentence in your conclusions.

In [ ]:
best = sorted(RESULTS, key=lambda r: r["mean"], reverse=HIGHER_IS_BETTER)[0]
print("Best on validation:", best["name"], best["changes"])

test_scores = []
for seed in SEEDS:
    keras.utils.set_random_seed(seed)
    model = build_model(best["config"])
    model.fit(x_train, y_train_s, validation_data=(x_val, y_val_s),
              epochs=best["config"]["epochs"], batch_size=best["config"]["batch_size"],
              callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=best["config"]["patience"],
                                                       restore_best_weights=True)],
              verbose=0)
    _, score = model.evaluate(x_test, y_test_raw / TARGET_SCALE, verbose=0)
    test_scores.append(score * TARGET_SCALE if TASK == "regression" else score)

print(f"TEST {METRIC}: {np.mean(test_scores):{FMT}} ± {np.std(test_scores, ddof=1):{FMT}}   (baseline was {BASELINE:{FMT}})")

## 8. Your conclusions

Write 3–5 sentences:
1. Which change helped the most? Which did nothing?
2. Did you see overfitting? Where (look at the loss curves)?
3. How far above the baseline did you get – and is it "good" for this problem?
4. What would you try next with more time?

## Stretch goals

- **CIFAR-10 / Fashion-MNIST:** replace the Dense layers with a small convnet from [Class 4](../../Class-4/Class-4.md) (reshape the input back to images!). How much better is it?
- **California housing:** compare `mse` and `huber` losses. Try predicting `log(price)` instead of the price.
- **Reuters:** use only the 2 000 most common words – how much accuracy do you lose? Report top-3 accuracy too.
- Switch the Keras backend to `"torch"` (or `"jax"`) at the top – do you get the same numbers?